In [1]:
import os
import sys
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.utils.data as data
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
# from sklearn.metrics import confusion_matrix, f1_score
# from sklearn.model_selection import train_test_split
import numpy as np

In [2]:
sys.path.append(os.path.abspath(os.path.join('..')))
from src.utils.data_loader import ButterflyDataset
from src.baseline.train import train_model
from src.baseline.evaluate import ClassificationEvaluator, LearningCurveTracker

In [3]:
BATCH_SIZE = 32
IMAGE_SIZE = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

path = "../data/raw"
img_dir = os.path.join(path, 'train')

full_train_df = pd.read_csv('splits/full_train_split.csv')
train_df = pd.read_csv('splits/train_split.csv')
val_df = pd.read_csv('splits/val_split.csv')
test_df = pd.read_csv('splits/test_split.csv')

train_dataset = ButterflyDataset(df=train_df, img_dir=img_dir, transform=data_transform)
train_dataloader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = ButterflyDataset(df=val_df, img_dir=img_dir, transform=data_transform)
val_dataloader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

full_train_dataset = ButterflyDataset(df=full_train_df, img_dir=img_dir, transform=data_transform)
full_train_dataloader = data.DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = ButterflyDataset(df=test_df, img_dir=img_dir, transform=data_transform)
test_dataloader = data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = sorted(test_df['label'].unique().tolist())
C = len(class_names)

In [ ]:
# train
model, history = train_model(
    data_loader=train_dataloader,
    # val_loader=val_dataloader,
    output_dir = '../src/baseline/',
    epochs=10,
    device=DEVICE,
    n_classes=C
)

TypeError: train_model() got an unexpected keyword argument 'output_file'

In [1]:
# test
model.eval()
all_preds = []
all_trues = []
with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_trues.extend(labels.cpu().numpy())

y_true = np.array(all_trues)
y_pred = np.array(all_preds)

NameError: name 'model' is not defined

In [ ]:
# --- Evaluator ---
ev = ClassificationEvaluator(class_names=class_names, n_classes=C)
ev.update(y_true, y_pred)
report = ev.compute()
ev.print_report(report)
ev.plot_confusion_matrix(report)


# --- Learning curves ---
tracker = LearningCurveTracker()
for epoch in range(len(history['train_loss'])):
    tracker.update(
        train_loss = history['train_loss'][epoch],
        train_acc  = history['train_acc'][epoch],
        val_loss   = history['val_loss'][epoch],
        val_acc    = history['val_acc'][epoch],
    )
print("Best epoch:", tracker.best_epoch("train_loss")) # Changed tracking target to train_loss since val is flat here
tracker.plot()